# House Prices — 고정 선형모델 + feature 엔지니어링 (Lost in Hyperspace 기법) (Colab)

**연습 기법** (IOAI 2024 *Lost in Hyperspace* 모범답안과 동일 원리): **모델은 고정(LinearRegression)** 해두고,
**오직 feature 엔지니어링**으로 회귀 RMSE 를 낮춘다 — '좋은 feature 가 좋은 모델보다 중요하다'.

| Lost in Hyperspace 모범답안2 (최고점 ≈1.52) | 이 노트북 |
|---|---|
| **고정 `LinearRegression`** | 동일 ✓ |
| feature 엔지니어링으로 RMSE↓ (before/after) | 동일 ✓ (Step1→Step2 시연) |
| log/RMSE 평가 | log(SalePrice) RMSE ✓ |
| PCA·채널통계·대칭(축순열)증강·타깃별 feature | ✗ (5×5×5×6 스펙트럼 배열 전용 — 표 데이터엔 부적용) |

**정직한 한계**: 모범답안 2 의 시그니처 feature(다채널 배열 PCA·채널통계·축순열 증강·타깃별 feature)는 *스펙트럼
배열* 구조에 특화돼 **표 데이터(house-prices)엔 적용 불가**(억지로 PCA 넣어도 도움 안 됨·타깃별은 단일타깃이라 N/A).
그래서 여긴 *고정 LinReg + feature 엔지니어링* 이라는 **핵심 원리**만 표 데이터용 feature(로그변환·범주형 더미·파생)로 재현.
(18처럼 데이터 종류 차이로 골격 전체는 못 맞추는 *정직한 한계* — 억지로 안 맞춤.)

**평가**: `log(SalePrice)` RMSE(낮을수록 좋음). `pandas`+`scikit-learn` 기본만.

## 0. 라이브러리 설치 & Kaggle 자격증명


In [ ]:
import sys, subprocess
for pkg in ["kaggle", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")


In [ ]:
import os

# Kaggle API 토큰 (내장)
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"

print("Kaggle 자격증명 설정 완료 (내장 토큰)")


## 1. Kaggle 에서 House Prices 데이터 다운로드


In [ ]:
import os, glob, zipfile

WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

# 대회 데이터 다운로드 + 압축 해제
api.competition_download_files("house-prices-advanced-regression-techniques", path=WORK_DIR, quiet=False)
for zpath in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(WORK_DIR)
    os.remove(zpath)

print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])


## 2. 데이터 준비 — 고정 모델 & 평가 지표
타깃은 `log(SalePrice)`(대회 지표가 로그 RMSE). **모델은 LinearRegression 으로 고정**하고, 이후 feature 만 바꾼다.


In [ ]:
import os, numpy as np, pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

train = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
y = np.log1p(train["SalePrice"].values)        # 대회 지표 = log RMSE
def rmse(a, b): return mean_squared_error(a, b) ** 0.5
print("train:", train.shape, " test:", test.shape)


## 3. Step 1 — 베이스라인: 숫자 feature 만
결측은 중앙값으로 채우고, 숫자형 컬럼만 그대로 사용.


In [ ]:
##########################
# Your code here
##########################

## 4. Step 2 — feature 엔지니어링 (모델은 그대로!)
- **공학 feature**: 총면적 `TotalSF`, 주택 나이 `Age`/`RemodAge`, 총 욕실 `TotalBath`
- **치우친 숫자 로그변환**(skew>1), **범주형 원핫 인코딩**(`get_dummies`)


In [ ]:
##########################
# Your code here
##########################

## 5. 최종 학습
전체 train 으로 다시 학습한 뒤 test 를 예측합니다 (로그 → 원래 가격으로 복원).

In [ ]:
##########################
# Your code here
##########################

## 6. 캐글 제출 파일 생성
예측을 `submission.csv` (Id, SalePrice) 로 저장합니다.

In [ ]:
submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"Id": test["Id"], "SalePrice": pred}).to_csv(submission_path, index=False)
print("Saved:", submission_path)
pd.read_csv(submission_path).head()

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)


## 🏁 제출하기
`submission.csv` 를 [House Prices](https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques) 에 제출하세요.

**기법 요약**: 모델(LinearRegression)을 **고정**하고 feature 만 설계해 RMSE 를 낮췄습니다 — IOAI *Lost in Hyperspace* 와 동일한 패턴.
더 낮추려면: 더 많은 공학 feature(면적·품질 상호작용), 이상치 제거, 표준화 후 PCA, 범주형 타깃 인코딩 등을 시도해 보세요.
